In [24]:
from langgraph.graph import StateGraph, START, END
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_huggingface import HuggingFaceEndpoint, ChatHuggingFace 
from typing import TypedDict, Literal
from langchain_core.messages import HumanMessage, SystemMessage
from pydantic import BaseModel, Field
from dotenv import load_dotenv
import os

load_dotenv()

api_key = os.getenv("GOOGLE_API_KEY")

In [25]:
gemini = ChatGoogleGenerativeAI(
    model= "gemini-2.5-pro",
    temperature=0.7,
    max_retries=2,
    max_tokens=1024,
    google_api_key=api_key,
)
gemini2 = ChatGoogleGenerativeAI(
    model= "gemini-2.5-flash",
    temperature=0.5,
    max_retries=2,
    max_tokens=1024,
    google_api_key=api_key
)

llm = HuggingFaceEndpoint(
    repo_id="mistralai/Mistral-7B-Instruct-v0.2",
    task="text-generation",
)

mistral = ChatHuggingFace(llm=llm)

In [26]:
class PostEvaluation(BaseModel):
    evaluation: Literal["approved", "needs_improvement"] = Field(..., description="Final evaluation result.")
    feedback: str = Field(..., description="feedback for the tweet.")

In [27]:
gemini_evaluter = gemini.with_structured_output(PostEvaluation)

In [28]:
class PostState(TypedDict):
    topic: str
    post: str
    feedback: str
    iteration: int
    max_iteration: int
    evaluation: Literal["approved", "needs_improvement"]

In [29]:
def post_generator(state:PostState) -> dict:
    messages = [
    SystemMessage(content="You are a knowledgeable and engaging LinkedIn educator and thought leader."),
    HumanMessage(content=f"""
Write a short, original, and insightful LinkedIn post on the topic: "{state['topic']}".  

Rules:
- Use a professional and approachable tone.
- Max 300 words.
- Focus on education, actionable insights, or lessons learned.
- Include relatable examples or analogies if appropriate.
- Avoid jokes, sarcasm, or overly casual language.
""")
]

    response = gemini2.invoke(messages).content
    return {'post': response}

In [30]:
def evaluater(state: PostState):
    # prompt
    messages = [
        SystemMessage(content="You are a discerning LinkedIn content critic. You evaluate posts based on clarity, value, engagement potential, and professional tone."),
        HumanMessage(content=f"""
Evaluate the following LinkedIn post:

Post: "{state['post']}"

Use the criteria below to evaluate the post:

1. Originality – Does it provide fresh insights or perspectives?  
2. Value – Does it teach something actionable or meaningful?  
3. Engagement Potential – Is it likely to spark comments, shares, or discussions?  
4. Clarity & Structure – Is it easy to read, well-organized, and professional in tone?  
5. Professionalism – Avoid casual jokes, sarcasm, or irrelevant content.

Auto-reject if:
- It contains unprofessional language, slang, or memes
- It is overly generic or vague without actionable advice
- It reads like a blog dump with no clear takeaway

### Respond ONLY in structured format:
- evaluation: "approved" or "needs_improvement"  
- feedback: One paragraph explaining the strengths and weaknesses 
""")
    ]

    response = gemini_evaluter.invoke(messages)

    return {'evaluation': response.evaluation, 'feedback': response.feedback}


In [31]:
def optimizer(state: PostState):
    messages = [
        SystemMessage(content="You refine LinkedIn posts to maximize clarity, engagement, and professional value based on given feedback."),
        HumanMessage(content=f"""
Improve the LinkedIn post based on this feedback:
"{state['feedback']}"

Topic: "{state['topic']}"
Original Post:
{state['post']}

Re-write it as a concise, professional, and insightful LinkedIn post. Make it easy to read, provide actionable value, and maximize engagement.
""")
    ]

    response = mistral.invoke(messages).content
    iteration = state['iteration'] + 1

    return {'post': response, 'iteration': iteration}

In [32]:
def router(state: PostState):
    evaluation = state['evaluation']
    iterations = state['iteration']
    if evaluation == "approved" or iterations>=state['max_iteration']:
        return "approved"
    else:
        return "needs_improvement"

In [33]:
#define the graph
graph = StateGraph(PostState)

#add nodes in the graph
graph.add_node("post_generator", post_generator)
graph.add_node("evaluater", evaluater)
graph.add_node("optimizer", optimizer)

#add edges
graph.add_edge(START, 'post_generator')
graph.add_edge('post_generator', 'evaluater')
graph.add_conditional_edges('evaluater', router, {"approved": END, "needs_improvement":'optimizer'})
graph.add_edge('optimizer', 'evaluater')

#compile the graph
workflow = graph.compile()

In [34]:
workflow.get_graph().print_ascii()

          +-----------+             
          | __start__ |             
          +-----------+             
                 *                  
                 *                  
                 *                  
        +----------------+          
        | post_generator |          
        +----------------+          
                 *                  
                 *                  
                 *                  
          +-----------+             
          | evaluater |             
          +-----------+             
          ...         ...           
         .               .          
       ..                 ..        
+---------+           +-----------+ 
| __end__ |           | optimizer | 
+---------+           +-----------+ 


In [38]:
initial_state = {
    'topic': "AI In INDIA",
    'iteration': 1,
    'max_iteration' : 2
}

In [39]:
final_state = workflow.invoke(initial_state)

Retrying langchain_google_genai.chat_models._chat_with_retry.<locals>._chat_with_retry in 2.0 seconds as it raised DeadlineExceeded: 504 Deadline Exceeded.


In [40]:
final_state['iteration']

2

In [41]:
final_state['evaluation']

'needs_improvement'

In [43]:
final_state

{'topic': 'AI In INDIA',
 'post': ' ## Unleashing India\'s Potential in AI: Bridging the Gap Between Innovation and Inclusion\n\nIndia is making bold strides in the AI sector, fueled by a burgeoning talent pool, abundant data resources, and advanced digital infrastructure, like UPI. These elements have positioned India as a formidable player in the global AI market.\n\nBut India\'s AI journey is about more than just technological innovation. It\'s also about leveraging this technology to bridge the gap between urban and rural regions, commonly referred to as "Bharat," and ensure responsible, inclusive growth.\n\nEmbrace the possibilities of AI in India and contribute to a future where technology empowers and connects all.\n\n[Insert a call-to-action, such as a thought-provoking question or invitation to join a group discussion, to drive engagement and further discussion.]\n\n#aiindia #technology #digitalinnovation #inclusion #responsibility',
 'feedback': 'The post is well-structured w